# Tine: run repository tests on recorded SWE-bench patches

**This notebook executes model-written code.** Use a fresh **CPU** runtime, run it, save the
outputs, then **Runtime → Disconnect and delete runtime**. Do not reuse the runtime for
anything else. Read `docs/SWE_CHECKS.md` in the repository first.

What it measures: of the wrong patches cheaper models wrote for SWE-bench tasks, how many
would a repository's own tests catch? That number decides whether try-cheap-then-check
routing can save money on real repositories.

How: Colab has no Docker, so the SWE-bench harness cannot run here as-is. The notebook replays
the harness's own install and test recipes natively, and first checks every task with its
reference patch. A task whose reference patch does not pass here is reported and skipped.

The work is unit tests on CPU cores. A GPU or TPU does not speed it up; more cores do. If your plan
offers it, pick **CPU** with **High-RAM** (more cores): the long run then takes roughly 1.5-2 hours
instead of 4-5.

Run all cells top to bottom. Cell 5 is a 10-minute smoke test: **stop and report if it fails.**
Cell 6 is the long run and can be rerun after a disconnect; it resumes.

In [ ]:
# 1. Mount Drive, get the repository, find the benchmark results.
from google.colab import drive
drive.mount("/content/drive")

import json, os, subprocess, sys
from pathlib import Path

BRANCH = "claude/avengers-pro-baseline"  # change to "main" once this branch is merged
ROOT = Path("/content/drive/MyDrive/tine")
OUT = ROOT / "swe"               # results kept on Drive (written by root only)
WORK = Path("/content/swe")      # scratch on the runtime's disk
REPO = Path("/content/tine")
OUT.mkdir(parents=True, exist_ok=True); WORK.mkdir(parents=True, exist_ok=True)
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                    "https://github.com/Nandu-Dammalapti/tine.git", str(REPO)], check=True)
os.chdir(REPO)
print("tine revision:", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "swebench==3.0.17", "huggingface-hub"], check=True)

manifest = ROOT / "benchmark" / "manifest.json"
if manifest.exists():
    BENCH = Path(json.loads(manifest.read_text())["bench_dir"])
else:  # first notebook not run on this Drive: fetch the pinned archive to the runtime disk
    subprocess.run([sys.executable, "scripts/fetch_benchmark.py", "--output-dir", str(WORK / "bench"),
                    "--revision", "b95870f9e60c589e82f7972c91ffa8de9ca89f85"], check=True)
    BENCH = WORK / "bench" / "extracted" / "bench-release"
SWE = BENCH / "swe-bench" / "verified"
print("models with SWE-bench answers:", sorted(p.name for p in SWE.iterdir() if p.is_dir()))

In [ ]:
# 2. Prepare the machine: conda (the harness's pinned version), an unprivileged test user,
#    and make sure that user cannot see Drive or /content.
RUN_USER = "tineswe"
if not Path("/opt/miniconda3/bin/conda").exists():
    subprocess.run("wget -q https://repo.anaconda.com/miniconda/Miniconda3-py311_23.11.0-2-Linux-x86_64.sh -O /tmp/miniconda.sh"
                   " && bash /tmp/miniconda.sh -b -p /opt/miniconda3"
                   " && /opt/miniconda3/bin/conda config --append channels conda-forge", shell=True, check=True)
subprocess.run("apt-get -qq update && apt-get -qq install -y build-essential libffi-dev libtiff-dev jq locales patch > /dev/null",
               shell=True, check=True)
subprocess.run(f"id {RUN_USER} >/dev/null 2>&1 || adduser --disabled-password --gecos '' {RUN_USER} >/dev/null",
               shell=True, check=True)
# Channel setting for every user and home directory (parallel workers use their own).
subprocess.run("/opt/miniconda3/bin/conda config --system --append channels conda-forge", shell=True)
os.chmod("/content", 0o700)  # the notebook runs as root; the test user gets no access to /content or Drive
pre = subprocess.run([sys.executable, "scripts/swe_repo_tests.py", "--preflight", "--run-user", RUN_USER,
                      "--protect", "/content", "/content/drive"], capture_output=True, text=True)
print(pre.stdout or pre.stderr)
assert pre.returncode == 0, "Preflight failed: do not continue. Report the block above."

In [ ]:
# 3. Task metadata and local mirrors of the 12 repositories.
from datasets import load_dataset
DATASET = WORK / "swe-bench-verified.jsonl"
if not DATASET.exists():
    load_dataset("princeton-nlp/SWE-bench_Verified", split="test").to_json(str(DATASET), lines=True)
REPOS = WORK / "repos"; REPOS.mkdir(exist_ok=True)
names = sorted({json.loads(line)["repo"] for line in DATASET.read_text().splitlines() if line.strip()})
for name in names:
    target = REPOS / (name.replace("/", "__") + ".git")
    if not target.exists():
        subprocess.run(["git", "clone", "-q", "--bare", f"https://github.com/{name}.git", str(target)], check=True)
print(len(names), "repositories mirrored;", subprocess.run(["du", "-sh", str(REPOS)], capture_output=True, text=True).stdout.split()[0])

In [ ]:
# 4. Static checks again, on Linux with GNU patch (nothing is executed; about 5 minutes).
STATIC = OUT / "static-checks-linux.csv"
if not STATIC.exists():
    subprocess.run([sys.executable, "scripts/swe_patch_checks.py", "--bench-dir", str(SWE),
                    "--metadata", str(DATASET), "--repos-dir", str(REPOS), "--output", str(STATIC),
                    "--workers", str(os.cpu_count() or 2)], check=True)
import pandas as pd
static = pd.read_csv(STATIC)
print("STATIC CHECKS (Linux):", static.reason.value_counts().to_dict())

In [ ]:
# 5. Smoke test: reference patches only, 3 django and 3 sympy tasks. About 10 minutes.
#    Every row should say status=ok and resolved=True. If not, STOP and report this output.
RESULTS = OUT / "repo-tests.csv"
common = [sys.executable, "scripts/swe_repo_tests.py", "--bench-dir", str(SWE), "--static-checks", str(STATIC),
          "--dataset", str(DATASET), "--mirror-dir", str(REPOS), "--run-user", RUN_USER,
          "--protect", "/content", "/content/drive", "--output", str(RESULTS)]
for repo in ["django/django", "sympy/sympy"]:
    subprocess.run([*common, "--repos", repo, "--limit-instances", "3", "--gold-only"], check=True)
smoke = pd.read_csv(RESULTS)
print(smoke[["instance_id", "model", "status", "resolved", "f2p_failed", "p2p_failed", "seconds"]].to_string(index=False))
for log in sorted(Path("/tmp/tine-swe-logs").glob("*.log"))[:2]:
    print("\n=====", log.name, "(last 60 lines)"); print("\n".join(log.read_text(errors="replace").splitlines()[-60:]))

In [ ]:
# 6. The long run. Resumable: rerun this cell after a disconnect (rerun cells 1-3 first).
#    Change MODELS to [] to run every model's answers (several times longer).
MODELS = ["claude-opus-4.1", "claude-sonnet-4", "deepseek-r1-0528", "deepseek-v3.1-terminus",
          "deepseek-v3-0324", "kimi-k2-0905", "glm-4.6", "qwen3-235b-a22b-2507"]
MAX_HOURS = "10"
WORKERS = str(os.cpu_count() or 2)  # one task per CPU core, each in its own environment
print("workers:", WORKERS)
subprocess.run([*common, "--repos", "django/django", "sympy/sympy", "--models", *MODELS,
                "--max-hours", MAX_HOURS, "--workers", WORKERS])

In [ ]:
# 7. Summary. Copy everything this cell prints back to Claude.
summary = subprocess.run([*common, "--summary"], capture_output=True, text=True)
print(summary.stdout or summary.stderr)
CHECKS = OUT / "repo-test-checks.csv"
print("CHECKS FILE:", subprocess.run([*common, "--checks-output", str(CHECKS)], capture_output=True, text=True).stdout.strip())
print("saved:", RESULTS, CHECKS, STATIC, sep="\n  ")
print("free disk GB:", round(__import__("shutil").disk_usage("/").free / 1e9, 1))

Done. Copy the printed summary, then choose **Runtime → Disconnect and delete runtime**.
Results are on Drive under `MyDrive/tine/swe/`: `repo-tests.csv` (one row per patch run),
`repo-test-checks.csv` (input for `tine cascade --checks`) and `static-checks-linux.csv`.